# 03 Python Unit Tests - pytest - podstawy
_Kamil Bartocha_ | _wersja 2.0_

## Rozkład jazdy

1. 🔹 **Filozofia pytest - prostota i czytelnosc** - funkcje, assert, discovery
2. 🔹 **Asercje i komunikaty bledow** - `pytest.raises`, `pytest.approx`
3. 🔹 **Uruchamianie i konfiguracja** - CLI, `pytest.ini`

🐍 Każda sekcja zawiera ćwiczenia.

## 1. 🔹 Filozofia pytest - prostota i czytelnosc

`pytest` to biblioteka do testowania, ktora upraszcza pisanie testow
w porownaniu z wbudowanym `unittest`. Wymaga instalacji:

```bash
pip install pytest
```

Kluczowe roznice w stosunku do `unittest`:

| Cecha | `unittest` | `pytest` |
|-------|-----------|----------|
| Import dla testow | `import unittest` | brak (opcjonalnie `import pytest`) |
| Struktura | klasa `TestCase` | zwykla funkcja |
| Asercje | `self.assertEqual(a, b)` | `assert a == b` |
| Wyjatki | `self.assertRaises(E)` | `pytest.raises(E)` |
| Floaty | `self.assertAlmostEqual(a, b)` | `assert a == pytest.approx(b)` |
| Pomijanie | `@unittest.skip(...)` | `@pytest.mark.skip(...)` |
| Uruchomienie | `python -m unittest` | `pytest` |

**Test discovery** - pytest automatycznie wykrywa testy jezeli:
- plik ma nazwe `test_*.py` lub `*_test.py`,
- funkcja lub metoda zaczyna sie od `test_`.

Najwazniejsza zaleta pytest: **przepisywanie asercji** (assert
rewriting). Przy niepowodzeniu pytest pokazuje dokladnie jakie
wartosci zawiodly, bez potrzeby pisania osobnych metod `assert*`:

```
AssertionError: assert 5 != 7
  where 5 = add(2, 3)
         where 7 = expected
```

> 💡 **Tip:** pytest jest kompatybilny z `unittest` - mozna
> uruchamiac testy napisane w obu stylach tym samym poleceniem
> `pytest`. Nie trzeba migrować całego projektu naraz.

In [ ]:
import pytest


# --- helper do uruchamiania testow pytest w notebooku ---
def _run(*test_fns):
    passed = failed = 0
    for fn in test_fns:
        try:
            fn()
            print(f"PASS  {fn.__name__}")
            passed += 1
        except AssertionError as e:
            print(f"FAIL  {fn.__name__}: {e or 'AssertionError'}")
            failed += 1
    print(f"\n{passed} passed, {failed} failed")


# CUT (Code Under Test)
class Calculator:
    def add(self, a, b):      return a + b
    def subtract(self, a, b): return a - b
    def multiply(self, a, b): return a * b
    def is_positive(self, n): return n > 0


# ----- unittest style (stary sposob) -----
import unittest

class TestCalculatorUnittest(unittest.TestCase):
    def test_add(self):
        calc = Calculator()
        self.assertEqual(calc.add(3, 4), 7)
    def test_is_positive(self):
        calc = Calculator()
        self.assertFalse(calc.is_positive(0))


# ----- pytest style (nowy sposob) -----
# Brak klasy, brak self, zwykly assert

def test_add_returns_correct_sum():
    calc = Calculator()
    assert calc.add(3, 4) == 7

def test_subtract_returns_correct_difference():
    calc = Calculator()
    assert calc.subtract(10, 3) == 7

def test_multiply_two_numbers():
    calc = Calculator()
    assert calc.multiply(4, 5) == 20

def test_is_positive_returns_false_for_zero():
    calc = Calculator()
    assert calc.is_positive(0) is False


_run(
    test_add_returns_correct_sum,
    test_subtract_returns_correct_difference,
    test_multiply_two_numbers,
    test_is_positive_returns_false_for_zero,
)

---

### 🐍 Ćwiczenia - filozofia pytest

**Ćw. 1.** Przetlumacz ponizsze testy `unittest` na styl `pytest`
(zwykle funkcje, `assert` zamiast metod `self.assert*`).

**Ćw. 2.** Napisz testy dla `fizzbuzz(n)` jako czyste funkcje
pytest. Pokryj minimum 5 przypadkow (liczby zwykle, wielokrotnosci
3, 5 i 15).

**Ćw. 10.** Ten sam test dla `is_palindrome()` napisz raz
w stylu `unittest` i raz w stylu `pytest`. Porownaj dlugosc
i czytelnosc.

In [ ]:
# Ćw. 1: przetlumacz testy unittest -> pytest
import pytest


class StringUtils:
    def reverse(self, text):         return text[::-1]
    def is_palindrome(self, text):   return text == text[::-1]
    def count_words(self, text):     return len(text.split())


# -- oryginalne testy unittest (nie zmieniaj) --
import unittest
class TestStringUtilsUnittest(unittest.TestCase):
    def test_reverse_hello(self):
        utils = StringUtils()
        self.assertEqual(utils.reverse("hello"), "olleh")
    def test_is_palindrome_racecar(self):
        utils = StringUtils()
        self.assertTrue(utils.is_palindrome("racecar"))
    def test_count_words_in_sentence(self):
        utils = StringUtils()
        self.assertEqual(utils.count_words("hello world foo"), 3)


# -- przetlumacz ponizej na styl pytest --

def test_reverse_hello():
    ...

def test_is_palindrome_racecar():
    ...

def test_count_words_in_sentence():
    ...


_run(test_reverse_hello, test_is_palindrome_racecar, test_count_words_in_sentence)

In [ ]:
# Ćw. 2: fizzbuzz jako czyste funkcje pytest
import pytest


def fizzbuzz(n):
    if n % 15 == 0: return "FizzBuzz"
    if n % 3  == 0: return "Fizz"
    if n % 5  == 0: return "Buzz"
    return str(n)


def test_fizzbuzz_regular_number():
    ...

def test_fizzbuzz_multiple_of_3_returns_fizz():
    ...

def test_fizzbuzz_multiple_of_5_returns_buzz():
    ...

def test_fizzbuzz_multiple_of_15_returns_fizzbuzz():
    ...

def test_fizzbuzz_one_returns_string_1():
    ...


_run(
    test_fizzbuzz_regular_number,
    test_fizzbuzz_multiple_of_3_returns_fizz,
    test_fizzbuzz_multiple_of_5_returns_buzz,
    test_fizzbuzz_multiple_of_15_returns_fizzbuzz,
    test_fizzbuzz_one_returns_string_1,
)

In [ ]:
# Ćw. 10: porownanie unittest vs pytest - ten sam test
import pytest
import unittest


def is_palindrome(text):
    return text == text[::-1]


# ----- wersja unittest -----
class TestIsPalindromeUnittest(unittest.TestCase):
    def test_racecar_is_palindrome(self):
        self.assertTrue(is_palindrome("racecar"))
    def test_hello_is_not_palindrome(self):
        self.assertFalse(is_palindrome("hello"))
    def test_empty_string_is_palindrome(self):
        self.assertTrue(is_palindrome(""))


# ----- wersja pytest (uzupelnij ponizej) -----
# hint: zapisz te same przypadki jako zwykle funkcje

def test_racecar_is_palindrome():
    ...

def test_hello_is_not_palindrome():
    ...

def test_empty_string_is_palindrome():
    ...


_run(
    test_racecar_is_palindrome,
    test_hello_is_not_palindrome,
    test_empty_string_is_palindrome,
)

# Obserwacja - porownaj:
# unittest: ... linii, wymaga: ...
# pytest:   ... linii, wymaga: ...

## 2. 🔹 Asercje i komunikaty bledow

**Przepisywanie asercji (assert rewriting)** - pytest analizuje
kod bajtowy testow i przy niepowodzeniu `assert` wyswietla
dokladne wartosci po obu stronach operatora. Przyklad bledu:

```
AssertionError: assert [1, 3, 5] == [1, 3, 6]
                                         ^
  where  [1, 3, 5] = sorted([3, 1, 5])
  and    [1, 3, 6] = expected_result
```

**`pytest.raises`** - testowanie wyjatkow (context manager):

```python
import pytest

def test_raises_value_error():
    with pytest.raises(ValueError):
        my_function(-1)
```

Aby sprawdzic komunikat wyjatku, uzywamy zmiennej `excinfo`:

```python
def test_error_message():
    with pytest.raises(ValueError) as excinfo:
        my_function(-1)
    assert "negative" in str(excinfo.value)
    # lub przez regex:
    # excinfo.match(r"negative \d+")
```

**`pytest.approx`** - porownywanie liczb zmiennoprzecinkowych:

```python
assert 0.1 + 0.2 == pytest.approx(0.3)          # domyslna tolerancja 1e-6
assert result    == pytest.approx(3.14, abs=0.01) # bezwzgledna tolerancja
assert ratio     == pytest.approx(0.5,  rel=1e-3) # wzgledna tolerancja
```

> 💡 **Tip:** `pytest.approx` dziala takze na listach i slownikach:
> `assert [0.1, 0.2] == pytest.approx([0.1, 0.2])`. Nie uzywaj
> zwyklego `round()` do porownan floatow w testach - maskuje
> rzeczywisty blad zaokraglenia.

In [ ]:
import math
import pytest


class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError(f"Amount must be positive, got {amount}")
        self.balance += amount
        return self.balance

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError(f"Amount must be positive, got {amount}")
        if amount > self.balance:
            raise ValueError(
                f"Insufficient funds: balance={self.balance}, "
                f"requested={amount}"
            )
        self.balance -= amount
        return self.balance


# --- Zwykly assert (pytest przepisuje na czytelny komunikat bledu) ---

def test_deposit_increases_balance():
    account = BankAccount(balance=100)
    account.deposit(50)
    assert account.balance == 150

def test_deposit_returns_new_balance():
    account = BankAccount(balance=100)
    result = account.deposit(25)
    assert result == 125


# --- pytest.raises - context manager ---

def test_deposit_negative_raises_value_error():
    account = BankAccount(balance=100)
    with pytest.raises(ValueError):
        account.deposit(-10)


# --- sprawdzenie komunikatu przez excinfo.value ---

def test_withdraw_insufficient_message_mentions_funds():
    account = BankAccount(balance=50)
    with pytest.raises(ValueError) as excinfo:
        account.withdraw(200)
    assert "Insufficient" in str(excinfo.value)


# --- pytest.approx dla floatow ---

def test_interest_calculation():
    principal = 1000.0
    rate = 0.035   # 3.5%
    interest = principal * rate
    assert interest == pytest.approx(35.0, rel=1e-6)

def test_circle_area():
    radius = 3.0
    area = math.pi * radius ** 2
    assert area == pytest.approx(28.274, abs=0.001)


_run(
    test_deposit_increases_balance,
    test_deposit_returns_new_balance,
    test_deposit_negative_raises_value_error,
    test_withdraw_insufficient_message_mentions_funds,
    test_interest_calculation,
    test_circle_area,
)

---

### 🐍 Ćwiczenia - asercje i komunikaty bledow

**Ćw. 3. *(Trudniejsze)*** Napisz testy dla `parse_date(text)`
sprawdzajac poprawne formaty i bledy dla niepoprawnych napisow.
Sprawdz komunikat wyjatku przez `excinfo.value`.

**Ćw. 5.** Napisz testy dla klasy `Temperature` pokrywajac
konwersje `to_fahrenheit()`, `to_kelvin()` i metody boolowskie.

**Ćw. 6.** Uzyj `pytest.raises` jako context manager do
przetestowania wyjatkow w `UserValidator.validate_age()`.

**Ćw. 7.** Sprawdz komunikat wyjatku przez `excinfo.value`
(lub `excinfo.match()`) w testach dla `divide(a, b)`.

**Ćw. 8.** Napisz testy dla `UserValidator.validate_username()`
obejmujace edge cases: pusty napis, za krotki, znaki specjalne.

**Ćw. 9.** Uzyj `pytest.approx` do testowania funkcji
geometrycznych zwracajacych wartosci zmiennoprzecinkowe.

In [ ]:
# Ćw. 3: parse_date - poprawne i niepoprawne formaty
import pytest
from datetime import date


def parse_date(text):
    """Parsuje napis 'YYYY-MM-DD' na obiekt date.
    Rzuca ValueError dla niepoprawnego formatu."""
    try:
        return date.fromisoformat(text)
    except (ValueError, TypeError) as e:
        raise ValueError(f"Invalid date format: {text!r}") from e


def test_parse_valid_date_returns_date_object():
    result = parse_date("2024-01-15")
    ...

def test_parse_valid_date_has_correct_year():
    ...

def test_parse_invalid_format_raises_value_error():
    # hint: uzyj pytest.raises
    ...

def test_parse_invalid_format_message_contains_input():
    # hint: uzyj excinfo.value aby sprawdzic komunikat
    with pytest.raises(ValueError) as excinfo:
        parse_date("15-01-2024")
    ...

def test_parse_none_raises_value_error():
    ...


_run(
    test_parse_valid_date_returns_date_object,
    test_parse_valid_date_has_correct_year,
    test_parse_invalid_format_raises_value_error,
    test_parse_invalid_format_message_contains_input,
    test_parse_none_raises_value_error,
)

In [ ]:
# Ćw. 5: testy dla klasy Temperature
import pytest


class Temperature:
    def __init__(self, celsius):
        if celsius < -273.15:
            raise ValueError(
                f"Temperature {celsius} below absolute zero"
            )
        self.celsius = celsius

    def to_fahrenheit(self):
        return self.celsius * 9 / 5 + 32

    def to_kelvin(self):
        return self.celsius + 273.15

    def is_freezing(self):
        return self.celsius <= 0

    def is_boiling(self):
        return self.celsius >= 100


def test_zero_celsius_to_fahrenheit_is_32():
    ...

def test_100_celsius_to_fahrenheit_is_212():
    ...

def test_zero_celsius_to_kelvin():
    # hint: uzyj pytest.approx
    ...

def test_zero_celsius_is_freezing():
    ...

def test_100_celsius_is_boiling():
    ...

def test_below_absolute_zero_raises():
    ...


_run(
    test_zero_celsius_to_fahrenheit_is_32,
    test_100_celsius_to_fahrenheit_is_212,
    test_zero_celsius_to_kelvin,
    test_zero_celsius_is_freezing,
    test_100_celsius_is_boiling,
    test_below_absolute_zero_raises,
)

In [ ]:
# Ćw. 6: pytest.raises jako context manager - validate_age()
import pytest


def validate_age(age):
    """Zwraca age jezeli poprawny [0..150], inaczej rzuca ValueError."""
    if not isinstance(age, int):
        raise TypeError(f"Age must be int, got {type(age).__name__}")
    if not 0 <= age <= 150:
        raise ValueError(f"Age {age} out of range [0, 150]")
    return age


def test_validate_age_valid_returns_age():
    assert validate_age(25) == 25

def test_validate_age_negative_raises_value_error():
    # hint: with pytest.raises(ValueError):
    ...

def test_validate_age_above_150_raises_value_error():
    ...

def test_validate_age_string_raises_type_error():
    ...

def test_validate_age_boundary_0_is_valid():
    ...

def test_validate_age_boundary_150_is_valid():
    ...


_run(
    test_validate_age_valid_returns_age,
    test_validate_age_negative_raises_value_error,
    test_validate_age_above_150_raises_value_error,
    test_validate_age_string_raises_type_error,
    test_validate_age_boundary_0_is_valid,
    test_validate_age_boundary_150_is_valid,
)

In [ ]:
# Ćw. 7: sprawdzenie komunikatu przez excinfo.value i excinfo.match()
import pytest


def divide(a, b):
    if not isinstance(b, (int, float)):
        raise TypeError(f"Divisor must be numeric, got {type(b).__name__}")
    if b == 0:
        raise ZeroDivisionError(f"Cannot divide {a} by zero")
    return a / b


def test_divide_valid():
    assert divide(10, 2) == 5.0

def test_divide_by_zero_raises():
    with pytest.raises(ZeroDivisionError):
        ...

def test_divide_by_zero_message_contains_zero():
    # hint: with pytest.raises(...) as excinfo:
    #       assert "zero" in str(excinfo.value)
    ...

def test_divide_by_zero_message_match_regex():
    # hint: excinfo.match(r"Cannot divide \d+")
    with pytest.raises(ZeroDivisionError) as excinfo:
        divide(10, 0)
    ...

def test_divide_string_divisor_raises_type_error():
    ...


_run(
    test_divide_valid,
    test_divide_by_zero_raises,
    test_divide_by_zero_message_contains_zero,
    test_divide_by_zero_message_match_regex,
    test_divide_string_divisor_raises_type_error,
)

In [ ]:
# Ćw. 8: UserValidator - edge cases dla validate_username()
import pytest


class UserValidator:
    MIN_LEN = 3
    MAX_LEN = 20

    def validate_username(self, username):
        """Zwraca True jezeli username jest poprawny:
        - dlugosc 3-20 znakow
        - tylko litery, cyfry i podkreslnik
        - nie zaczyna sie od cyfry
        """
        if not isinstance(username, str):
            raise TypeError("Username must be a string")
        if len(username) < self.MIN_LEN:
            raise ValueError(
                f"Username too short (min {self.MIN_LEN})"
            )
        if len(username) > self.MAX_LEN:
            raise ValueError(
                f"Username too long (max {self.MAX_LEN})"
            )
        if not username.replace("_", "").isalnum():
            raise ValueError("Username contains invalid characters")
        if username[0].isdigit():
            raise ValueError("Username cannot start with a digit")
        return True


def test_valid_username_returns_true():
    ...

def test_empty_username_raises():
    ...

def test_too_short_username_raises():
    ...

def test_too_long_username_raises():
    ...

def test_special_chars_raises():
    # hint: np. "user@name"
    ...

def test_username_starting_with_digit_raises():
    ...

def test_username_with_underscore_is_valid():
    ...


_run(
    test_valid_username_returns_true,
    test_empty_username_raises,
    test_too_short_username_raises,
    test_too_long_username_raises,
    test_special_chars_raises,
    test_username_starting_with_digit_raises,
    test_username_with_underscore_is_valid,
)

In [ ]:
# Ćw. 9: pytest.approx dla obliczen geometrycznych
import math
import pytest


def circle_area(r):       return math.pi * r ** 2
def circle_perimeter(r):  return 2 * math.pi * r
def triangle_area(b, h):  return 0.5 * b * h
def hypotenuse(a, b):     return math.sqrt(a ** 2 + b ** 2)


def test_circle_area_r1():
    # hint: math.pi * 1**2 = 3.14159...
    assert circle_area(1) == pytest.approx(...)

def test_circle_perimeter_r5():
    # hint: 2 * math.pi * 5 = 31.4159...
    ...

def test_triangle_area():
    ...

def test_hypotenuse_3_4_is_5():
    # hint: uzyj abs= lub rel= tolerancji
    ...

def test_float_precision_without_approx_fails():
    # Ten test pokazuje problem z porownywaniem floatow:
    # 0.1 + 0.2 != 0.3 w arytmetyce IEEE 754
    assert 0.1 + 0.2 == pytest.approx(0.3)


_run(
    test_circle_area_r1,
    test_circle_perimeter_r5,
    test_triangle_area,
    test_hypotenuse_3_4_is_5,
    test_float_precision_without_approx_fails,
)

## 3. 🔹 Uruchamianie i konfiguracja

**Najwazniejsze flagi CLI `pytest`:**

| Flaga | Dzialanie |
|-------|-----------|
| `pytest` | odkryj i uruchom wszystkie testy |
| `pytest test_foo.py` | uruchom konkretny plik |
| `pytest -v` | verbose - pokaz nazwe kazdego testu |
| `pytest -q` | quiet - minimalny output |
| `pytest -k "add or subtract"` | filtruj testy po nazwie |
| `pytest --tb=short` | skrócony traceback przy bledzie |
| `pytest --tb=no` | bez traceback |
| `pytest -x` | zatrzymaj po pierwszym bledzie |
| `pytest --co` | tylko wylistuj testy (nie uruchamiaj) |
| `pytest -s` | pokazuj print() z testow |

**Konfiguracja `pytest.ini`** (lub `pyproject.toml`):

```ini
[pytest]
testpaths = tests
addopts   = -v --tb=short
```

Najczesciej uzywane opcje w `pytest.ini`:

| Klucz | Opis |
|-------|------|
| `testpaths` | katalogi z testami (domyslnie `.`) |
| `addopts` | domyslne flagi CLI |
| `python_files` | wzorzec plikow testowych |
| `python_classes` | wzorzec klas testowych |
| `python_functions` | wzorzec funkcji testowych |
| `filterwarnings` | filtrowanie ostrzezen |

> 💡 **Tip:** `addopts = -v --tb=short -x` to dobry punkt
> startowy dla wiekszosci projektow - verbose, krotki traceback
> i zatrzymanie przy pierwszym bledzie przyspieszaja cykl TDD.

In [ ]:
import subprocess
import sys
import tempfile
import os


# Przyklad: uruchamiamy pytest na tymczasowym pliku z testami

test_content = """\
def test_add():       assert 1 + 1 == 2
def test_subtract():  assert 5 - 3 == 2
def test_multiply():  assert 3 * 4 == 12
def test_failing():   assert 1 == 2       # celowy blad
"""

tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix="_test.py", delete=False, dir="."
)
tmp.write(test_content)
tmp.close()


def _pytest(flags, label):
    print(f"\n=== pytest {' '.join(flags)} ===")
    result = subprocess.run(
        [sys.executable, "-m", "pytest", tmp.name] + flags,
        capture_output=True, text=True
    )
    print(result.stdout[-800:])  # ostatnie 800 znakow outputu


_pytest(["-v"],              "-v (verbose)")
_pytest(["-q"],              "-q (quiet)")
_pytest(["-k", "add or sub"], "-k 'add or sub'")
_pytest(["--tb=short", "-v"], "--tb=short")
_pytest(["--co", "-q"],       "--co (collect only)")

os.unlink(tmp.name)

---

### 🐍 Ćwiczenia - uruchamianie i konfiguracja

**Ćw. 4.** Uruchom testy ponizej uzywajac `subprocess` lub
terminala z flagami: `-v`, `-k "deposit"`, `--tb=short`, `-x`.
Zapisz wyniki w komentarzach.

**Ćw. 11. *(Trudniejsze)*** Napisz kompletne testy dla klasy
`ShoppingCart` pokrywajac metody `add_item`, `remove_item`,
`total` i `apply_discount`, w tym wyjatki i edge cases.

**Ćw. 12. *(Trudniejsze)*** Uzupelnij szablon `pytest.ini`
podajac odpowiednie wartosci. Nastepnie zapisz go do pliku
i uruchom testy, aby sprawdzic, ze konfiguracja dziala.

In [ ]:
# Ćw. 4: pytest CLI - uruchom z roznymi flagami
import subprocess
import sys
import tempfile
import os


# Ponizsze testy sa przygotowane - uruchom je z roznymi flagami
test_content = """
import pytest

class BankAccount:
    def __init__(self, balance=0): self.balance = balance
    def deposit(self, amount):
        if amount <= 0: raise ValueError("Must be positive")
        self.balance += amount
    def withdraw(self, amount):
        if amount > self.balance: raise ValueError("Insufficient")
        self.balance -= amount

def test_deposit_positive():      a = BankAccount(100); a.deposit(50);   assert a.balance == 150
def test_deposit_negative_raises(): 
    a = BankAccount(100)
    with pytest.raises(ValueError): a.deposit(-1)
def test_withdraw_valid():        a = BankAccount(100); a.withdraw(30);  assert a.balance == 70
def test_withdraw_too_much_raises():
    a = BankAccount(50)
    with pytest.raises(ValueError): a.withdraw(200)
def test_initial_balance_zero():  assert BankAccount().balance == 0
"""

tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix="_test.py", delete=False, dir="."
)
tmp.write(test_content)
tmp.close()

# Flaga -v
print("=== -v ===")
result = subprocess.run(
    [sys.executable, "-m", "pytest", tmp.name, "-v"],
    capture_output=True, text=True
)
print(result.stdout)

# Uzupelnij: flaga -k "deposit"
print("=== -k 'deposit' ===")
...  # hint: zmien flagi w subprocess.run

# Uzupelnij: flaga -x (zatrzymaj po 1. bledzie)
print("=== -x ===")
...

os.unlink(tmp.name)

In [ ]:
# Ćw. 11: kompletne testy dla ShoppingCart
import pytest


class ShoppingCart:
    def __init__(self):
        self.items = []

    def add_item(self, name, price, qty=1):
        if price < 0:
            raise ValueError(f"Price cannot be negative: {price}")
        if qty < 1:
            raise ValueError(f"Quantity must be at least 1: {qty}")
        self.items.append({"name": name, "price": price, "qty": qty})

    def remove_item(self, name):
        self.items = [i for i in self.items if i["name"] != name]

    def total(self):
        return sum(i["price"] * i["qty"] for i in self.items)

    def apply_discount(self, percent):
        if not 0 < percent <= 100:
            raise ValueError(
                f"Discount must be between 1 and 100, got {percent}"
            )
        return self.total() * (1 - percent / 100)

    def is_empty(self):
        return len(self.items) == 0


# --- add_item ---
def test_add_item_increases_item_count():
    ...

def test_add_item_negative_price_raises():
    ...

def test_add_item_zero_qty_raises():
    ...

# --- remove_item ---
def test_remove_item_empties_cart():
    ...

def test_remove_nonexistent_item_does_not_raise():
    # hint: nie rzuca wyjatku - brak elementu jest OK
    ...

# --- total ---
def test_total_empty_cart_is_zero():
    ...

def test_total_with_multiple_items():
    ...

def test_total_respects_quantity():
    ...

# --- apply_discount ---
def test_apply_discount_50_percent():
    # hint: uzyj pytest.approx
    ...

def test_apply_discount_zero_raises():
    ...

def test_apply_discount_above_100_raises():
    ...


_run(
    test_add_item_increases_item_count,
    test_add_item_negative_price_raises,
    test_add_item_zero_qty_raises,
    test_remove_item_empties_cart,
    test_remove_nonexistent_item_does_not_raise,
    test_total_empty_cart_is_zero,
    test_total_with_multiple_items,
    test_total_respects_quantity,
    test_apply_discount_50_percent,
    test_apply_discount_zero_raises,
    test_apply_discount_above_100_raises,
)

In [ ]:
# Ćw. 12: konfiguracja pytest.ini
import os
import subprocess
import sys
import tempfile


# Uzupelnij konfiguracje pytest.ini - zastap ... wlasciwymi wartosciami
pytest_ini = """\
[pytest]
testpaths   = ...
addopts     = ... --tb=short
python_files     = test_*.py
python_classes   = Test*
python_functions = test_*
"""
# hint: testpaths - katalog z testami (np. . lub tests)
# hint: addopts - dodaj -v aby zawsze wyswietlac nazwy testow

test_code = """\
def test_one():   assert 1 + 1 == 2
def test_two():   assert "pytest".upper() == "PYTEST"
def test_three(): assert [1, 2, 3][0] == 1
"""

# Zapisz konfiguracje i testy do katalogu tymczasowego
tmpdir = tempfile.mkdtemp()
with open(os.path.join(tmpdir, "pytest.ini"), "w") as f:
    f.write(pytest_ini)
with open(os.path.join(tmpdir, "test_sample.py"), "w") as f:
    f.write(test_code)

# Uruchom pytest z katalogu z konfiguracja
result = subprocess.run(
    [sys.executable, "-m", "pytest"],
    cwd=tmpdir,
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr[:300] if result.stderr else "")

# Wyswietl zawartosc pytest.ini
print("\n--- pytest.ini ---")
print(pytest_ini)